# Information Leakage Detection

Pre-trade baselines reveal whether the mid is already moving before a trade
executes — the signature of informed flow, seen from the maker's perspective.

Data: Binance spot BTCUSDT, 2026-03-01 (first 6 hours).

In [ ]:
import polars as pl
from fetch_data import fetch_binance_aggtrades, prepare_binance_trades_and_quotes

import markoutlib as mo

## Download and prepare data

In [ ]:
raw = fetch_binance_aggtrades("BTCUSDT", "2026-03-01")
trades, quotes = prepare_binance_trades_and_quotes(raw)

# Use first 6 hours to keep the 61-horizon analysis tractable.
cutoff = trades["timestamp"].min() + pl.duration(hours=6)
trades = trades.filter(pl.col("timestamp") <= cutoff)
quotes = quotes.filter(pl.col("timestamp") <= cutoff + pl.duration(minutes=5))

print(f"Trades: {trades.shape[0]:,}")
print(f"Quotes: {quotes.shape[0]:,}")

## Compute pre-trade and post-trade markouts

61 horizons from -30 seconds to +30 seconds, stepping by 1 second.
Negative horizons look backward: where was the mid *before* the trade?
From the maker's perspective, a positive pre-trade markout means the
mid was already moving against the maker before the fill.

In [ ]:
result = mo.compute(
    trades=trades,
    quotes=quotes,
    horizons=mo.seconds_range(-30, 30, 1),
    unit="bps",
    perspective="maker",
)

## The crossing-zero curve

The x-axis crosses zero at the moment of the trade.
Left of zero shows the pre-trade mid trajectory; right of zero
is the standard post-trade markout. If the curve is already
trending before the trade, information is leaking into the
mid ahead of execution.

In [ ]:
result.plot.curve()

There is clear information leakage: the maker's markout is already +1.7 bps at -30 seconds and steadily worsens toward the trade. The mid is trending against the maker well before the fill arrives. After the trade, the markout drops sharply to -1.5 bps at +1 second and continues to -2.0 bps at +30 seconds. The asymmetry around zero — the sharp discontinuity at the trade — is the informed component: information that was building up pre-trade gets confirmed post-trade.

## Segment by trade size

Does the pre-trade signal differ by trade size? Larger trades
from informed participants might show a steeper pre-trade ramp.

In [ ]:
q33 = trades["size"].quantile(0.33)
q67 = trades["size"].quantile(0.67)

trades_tagged = trades.with_columns(
    pl.when(pl.col("size") <= q33)
    .then(pl.lit("small"))
    .when(pl.col("size") <= q67)
    .then(pl.lit("medium"))
    .otherwise(pl.lit("large"))
    .alias("size_bucket")
)

result_sized = mo.compute(
    trades=trades_tagged,
    quotes=quotes,
    horizons=mo.seconds_range(-30, 30, 1),
    unit="bps",
    perspective="maker",
)

result_sized.plot.curve(by="size_bucket")

## Quantify the pre-trade ramp

Compare the maker's markout at -30s, -10s, -5s, and 0 for each size bucket.
A steeper decline from -30 to 0 means more information is leaking before the trade.

In [ ]:
curve = result_sized.curve(by="size_bucket")
pre_trade = curve.filter(pl.col("horizon_value").is_in([-30.0, -10.0, -5.0, 0.0]))
print(pre_trade.select("size_bucket", "horizon_value", "markout_mean", "n_obs"))

Large trades show the strongest pre-trade signal (+2.2 bps at -30s) and the deepest post-trade loss (-2.2 at +30s), giving a total swing of ~4.4 bps — the clearest signature of informed flow. Small trades follow a similar pattern but with slightly less pre-trade buildup. Medium trades are the least informed, consistent with the markout analysis in notebook 02.

## Heatmap: size bucket x horizon

In [ ]:
result_sized.plot.heatmap(by="size_bucket")